## Import libs

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.python.data import Dataset
from tensorflow.keras.optimizers import Adam
import seaborn as sns


from falsb4mpa.modeling.zhang.learning.multi_adv import train_loop as zhang_train
from falsb4mpa.dataset.load_data import load_data
from falsb4mpa.evaluation.evaluation import compute_predictive_metrics, compute_fair_metrics, compute_adv_metrics, compute_tradeoff, fair_evaluation, compute_intersectional_fair_metrics
from falsb4mpa.modeling.zhang.models.multi_adv import ZhangMultAdv

## Preliminaries

In [2]:
batch_size = 64
epochs = 100
learning_rate = 0.001

In [3]:
# cv_seeds = [13]
cv_seeds = [13, 29, 42, 55, 73]

## Load data

In [4]:
data_name = 'compas-mpa-cat-wout-agg'

In [5]:
x, y, a1, a2 = load_data(data_name)
raw_data = (x, y, a1, a2)

In [6]:
xdim = x.shape[1]
ydim = y.shape[1]
a1dim = a1.shape[1]
a2dim = a2.shape[1]
zdim = 8
print(xdim, ydim, a1dim, a2dim, zdim)

18 1 1 6 8


## Result file

In [ ]:
header = [
    "model_name", "cv_seed", 
    "clas_acc", "f1-micro", "f1-macro",
    "a1_dp", "a1_deqodds", "a1_deqopp", 
    "a2_dp", "a2_deqodds", "a2_deqopp",
    "wc_spd", "wc_aod", "wc_eod",
    "last_cosine_similarity"
]

results = []

## Testing

### DemPar

In [8]:
fairdef = "DemPar"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvBin4DP', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp]
    result += [a2_dp, a2_deqodds, a2_deqopp]
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

2026-05-15 14:07:49.640153: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 1 | Clf loss/acc 0.79/0.28 | Adv1 loss/acc 0.56/0.81 | Adv2 loss/acc 1.74/0.51 | Cos Sim 0.00


2026-05-15 14:07:52.077035: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 2 | Clf loss/acc 0.69/0.37 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.72/0.51 | Cos Sim -0.04
> Epoch: 3 | Clf loss/acc 0.63/0.60 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.71/0.51 | Cos Sim -0.13


2026-05-15 14:07:56.986104: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


...

> Epoch: 1 | Clf loss/acc 0.84/0.28 | Adv1 loss/acc 0.56/0.80 | Adv2 loss/acc 1.74/0.51 | Cos Sim -0.01
> Epoch: 2 | Clf loss/acc 0.70/0.38 | Adv1 loss/acc 0.56/0.80 | Adv2 loss/acc 1.73/0.51 | Cos Sim -0.10
> Epoch: 3 | Clf loss/acc 0.62/0.59 | Adv1 loss/acc 0.56/0.80 | Adv2 loss/acc 1.72/0.51 | Cos Sim -0.19
> Epoch: 4 | Clf loss/acc 0.57/0.68 | Adv1 loss/acc 0.56/0.80 | Adv2 loss/acc 1.71/0.51 | Cos Sim -0.23
> Epoch: 5 | Clf loss/acc 0.54/0.70 | Adv1 loss/acc 0.56/0.80 | Adv2 loss/acc 1.69/0.51 | Cos Sim -0.23
> Epoch: 6 | Clf loss/acc 0.52/0.72 | Adv1 loss/acc 0.56/0.80 | Adv2 loss/acc 1.67/0.51 | Cos Sim -0.22
> Epoch: 7 | Clf loss/acc 0.51/0.73 | Adv1 loss/acc 0.56/0.80 | Adv2 loss/acc 1.65/0.51 | Cos Sim -0.20
> Epoch: 8 | Clf loss/acc 0.50/0.74 | Adv1 loss/acc 0.56/0.80 | Adv2 loss/acc 1.63/0.51 | Cos Sim -0.19
> Epoch: 9 | Clf loss/acc 0.50/0.75 | Adv1 loss/acc 0.56/0.80 | Adv2 loss/acc 1.61/0.51 | Cos Sim -0.18
> Epoch: 10 | Clf loss/acc 0.49/0.75 | Adv1 loss/acc 0.56/0.80 |

2026-05-15 14:18:12.081239: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 54 | Clf loss/acc 0.51/0.77 | Adv1 loss/acc 0.64/0.80 | Adv2 loss/acc 1.20/0.51 | Cos Sim -0.21
> Epoch: 55 | Clf loss/acc 0.51/0.77 | Adv1 loss/acc 0.64/0.80 | Adv2 loss/acc 1.20/0.51 | Cos Sim -0.21
> Epoch: 56 | Clf loss/acc 0.51/0.77 | Adv1 loss/acc 0.65/0.80 | Adv2 loss/acc 1.20/0.51 | Cos Sim -0.21
> Epoch: 57 | Clf loss/acc 0.51/0.77 | Adv1 loss/acc 0.65/0.80 | Adv2 loss/acc 1.20/0.51 | Cos Sim -0.21
> Epoch: 58 | Clf loss/acc 0.51/0.77 | Adv1 loss/acc 0.65/0.80 | Adv2 loss/acc 1.19/0.51 | Cos Sim -0.21
> Epoch: 59 | Clf loss/acc 0.51/0.77 | Adv1 loss/acc 0.65/0.80 | Adv2 loss/acc 1.19/0.51 | Cos Sim -0.21
> Epoch: 60 | Clf loss/acc 0.51/0.77 | Adv1 loss/acc 0.65/0.80 | Adv2 loss/acc 1.19/0.51 | Cos Sim -0.22
> Epoch: 61 | Clf loss/acc 0.51/0.77 | Adv1 loss/acc 0.66/0.80 | Adv2 loss/acc 1.19/0.51 | Cos Sim -0.22
> Epoch: 62 | Clf loss/acc 0.51/0.77 | Adv1 loss/acc 0.66/0.80 | Adv2 loss/acc 1.19/0.51 | Cos Sim -0.22
> Epoch: 63 | Clf loss/acc 0.51/0.77 | Adv1 loss/acc 0.

/Users/lffpl/Projects/falsb4mpa/falsb4mpa/evaluation/baseline_metrics.py:41: RuntimeWarning: invalid value encountered in scalar divide
  return pos(Y) / (pos(Y) + neg(Y))


> Epoch: 1 | Clf loss/acc 0.83/0.27 | Adv1 loss/acc 0.56/0.81 | Adv2 loss/acc 1.74/0.51 | Cos Sim 0.00
> Epoch: 2 | Clf loss/acc 0.72/0.37 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.72/0.51 | Cos Sim -0.06
> Epoch: 3 | Clf loss/acc 0.65/0.60 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.71/0.51 | Cos Sim -0.12
> Epoch: 4 | Clf loss/acc 0.61/0.69 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.70/0.51 | Cos Sim -0.16
> Epoch: 5 | Clf loss/acc 0.57/0.70 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.68/0.51 | Cos Sim -0.16
> Epoch: 6 | Clf loss/acc 0.55/0.72 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.66/0.51 | Cos Sim -0.14
> Epoch: 7 | Clf loss/acc 0.53/0.72 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.63/0.51 | Cos Sim -0.13
> Epoch: 8 | Clf loss/acc 0.52/0.73 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.61/0.51 | Cos Sim -0.13
> Epoch: 9 | Clf loss/acc 0.51/0.74 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.58/0.51 | Cos Sim -0.12
> Epoch: 10 | Clf loss/acc 0.50/0.75 | Adv1 loss/acc 0.57/0.81 | 

### EqOdds

In [ ]:
fairdef = "EqOdds"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvBin4EqOdds', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp]
    result += [a2_dp, a2_deqodds, a2_deqopp]
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

> Epoch: 1 | Clf loss/acc 0.79/0.28 | Adv1 loss/acc 0.56/0.81 | Adv2 loss/acc 1.69/0.51 | Cos Sim 0.00
> Epoch: 2 | Clf loss/acc 0.69/0.37 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.65/0.51 | Cos Sim -0.02
> Epoch: 3 | Clf loss/acc 0.63/0.59 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.64/0.51 | Cos Sim -0.09
> Epoch: 4 | Clf loss/acc 0.60/0.68 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.61/0.51 | Cos Sim -0.13
> Epoch: 5 | Clf loss/acc 0.58/0.70 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.58/0.51 | Cos Sim -0.13
> Epoch: 6 | Clf loss/acc 0.57/0.72 | Adv1 loss/acc 0.56/0.81 | Adv2 loss/acc 1.55/0.51 | Cos Sim -0.13
> Epoch: 7 | Clf loss/acc 0.56/0.72 | Adv1 loss/acc 0.56/0.81 | Adv2 loss/acc 1.52/0.51 | Cos Sim -0.11
> Epoch: 8 | Clf loss/acc 0.56/0.73 | Adv1 loss/acc 0.56/0.81 | Adv2 loss/acc 1.49/0.51 | Cos Sim -0.09
> Epoch: 9 | Clf loss/acc 0.55/0.74 | Adv1 loss/acc 0.56/0.81 | Adv2 loss/acc 1.46/0.51 | Cos Sim -0.08
> Epoch: 10 | Clf loss/acc 0.55/0.75 | Adv1 loss/acc 0.56/0.81 | 

/Users/lffpl/Projects/falsb4mpa/falsb4mpa/evaluation/baseline_metrics.py:41: RuntimeWarning: invalid value encountered in scalar divide
  return pos(Y) / (pos(Y) + neg(Y))


> WC_SPD | WC_AOD | WC_EOD
> 0.25 | 0.1666666567325592 | 0.0
> Epoch: 1 | Clf loss/acc 0.83/0.28 | Adv1 loss/acc 0.59/0.80 | Adv2 loss/acc 1.68/0.51 | Cos Sim 0.09
> Epoch: 2 | Clf loss/acc 0.72/0.36 | Adv1 loss/acc 0.60/0.80 | Adv2 loss/acc 1.65/0.51 | Cos Sim 0.07
> Epoch: 3 | Clf loss/acc 0.65/0.60 | Adv1 loss/acc 0.59/0.80 | Adv2 loss/acc 1.64/0.51 | Cos Sim 0.04
> Epoch: 4 | Clf loss/acc 0.60/0.68 | Adv1 loss/acc 0.59/0.80 | Adv2 loss/acc 1.62/0.51 | Cos Sim 0.03
> Epoch: 5 | Clf loss/acc 0.57/0.70 | Adv1 loss/acc 0.59/0.80 | Adv2 loss/acc 1.59/0.51 | Cos Sim 0.03
> Epoch: 6 | Clf loss/acc 0.54/0.71 | Adv1 loss/acc 0.58/0.80 | Adv2 loss/acc 1.55/0.51 | Cos Sim 0.04
> Epoch: 7 | Clf loss/acc 0.52/0.72 | Adv1 loss/acc 0.58/0.80 | Adv2 loss/acc 1.52/0.51 | Cos Sim 0.04
> Epoch: 8 | Clf loss/acc 0.51/0.73 | Adv1 loss/acc 0.57/0.80 | Adv2 loss/acc 1.48/0.51 | Cos Sim 0.05
> Epoch: 9 | Clf loss/acc 0.50/0.74 | Adv1 loss/acc 0.57/0.80 | Adv2 loss/acc 1.44/0.51 | Cos Sim 0.05
> Epoch: 10 

/Users/lffpl/Projects/falsb4mpa/falsb4mpa/evaluation/baseline_metrics.py:41: RuntimeWarning: invalid value encountered in scalar divide
  return pos(Y) / (pos(Y) + neg(Y))


> WC_SPD | WC_AOD | WC_EOD
> 0.0 | 0.0 | 0.0
> Epoch: 1 | Clf loss/acc 0.84/0.28 | Adv1 loss/acc 0.56/0.80 | Adv2 loss/acc 1.70/0.51 | Cos Sim -0.01
> Epoch: 2 | Clf loss/acc 0.70/0.38 | Adv1 loss/acc 0.57/0.80 | Adv2 loss/acc 1.67/0.51 | Cos Sim -0.08
> Epoch: 3 | Clf loss/acc 0.62/0.59 | Adv1 loss/acc 0.57/0.80 | Adv2 loss/acc 1.65/0.51 | Cos Sim -0.15
> Epoch: 4 | Clf loss/acc 0.58/0.68 | Adv1 loss/acc 0.56/0.80 | Adv2 loss/acc 1.63/0.51 | Cos Sim -0.18
> Epoch: 5 | Clf loss/acc 0.55/0.70 | Adv1 loss/acc 0.56/0.80 | Adv2 loss/acc 1.60/0.51 | Cos Sim -0.17
> Epoch: 6 | Clf loss/acc 0.53/0.72 | Adv1 loss/acc 0.56/0.80 | Adv2 loss/acc 1.57/0.51 | Cos Sim -0.15
> Epoch: 7 | Clf loss/acc 0.51/0.73 | Adv1 loss/acc 0.55/0.80 | Adv2 loss/acc 1.54/0.51 | Cos Sim -0.14
> Epoch: 8 | Clf loss/acc 0.50/0.74 | Adv1 loss/acc 0.55/0.80 | Adv2 loss/acc 1.51/0.51 | Cos Sim -0.12
> Epoch: 9 | Clf loss/acc 0.50/0.74 | Adv1 loss/acc 0.55/0.80 | Adv2 loss/acc 1.48/0.51 | Cos Sim -0.11
> Epoch: 10 | Clf l

/Users/lffpl/Projects/falsb4mpa/falsb4mpa/evaluation/baseline_metrics.py:41: RuntimeWarning: invalid value encountered in scalar divide
  return pos(Y) / (pos(Y) + neg(Y))


> WC_SPD | WC_AOD | WC_EOD
> 0.0 | 0.0 | 0.0
> Epoch: 1 | Clf loss/acc 0.83/0.27 | Adv1 loss/acc 0.56/0.81 | Adv2 loss/acc 1.68/0.51 | Cos Sim -0.00
> Epoch: 2 | Clf loss/acc 0.72/0.37 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.64/0.51 | Cos Sim -0.05
> Epoch: 3 | Clf loss/acc 0.66/0.60 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.63/0.51 | Cos Sim -0.10
> Epoch: 4 | Clf loss/acc 0.61/0.68 | Adv1 loss/acc 0.57/0.81 | Adv2 loss/acc 1.61/0.51 | Cos Sim -0.12
> Epoch: 5 | Clf loss/acc 0.57/0.70 | Adv1 loss/acc 0.56/0.81 | Adv2 loss/acc 1.58/0.51 | Cos Sim -0.11
> Epoch: 6 | Clf loss/acc 0.55/0.71 | Adv1 loss/acc 0.56/0.81 | Adv2 loss/acc 1.54/0.51 | Cos Sim -0.10
> Epoch: 7 | Clf loss/acc 0.53/0.73 | Adv1 loss/acc 0.56/0.81 | Adv2 loss/acc 1.50/0.51 | Cos Sim -0.08
> Epoch: 8 | Clf loss/acc 0.52/0.73 | Adv1 loss/acc 0.56/0.81 | Adv2 loss/acc 1.45/0.51 | Cos Sim -0.07
> Epoch: 9 | Clf loss/acc 0.51/0.74 | Adv1 loss/acc 0.56/0.81 | Adv2 loss/acc 1.41/0.51 | Cos Sim -0.07
> Epoch: 10 | Clf l

2026-05-15 17:38:34.005052: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 15 | Clf loss/acc 0.44/0.76 | Adv1 loss/acc 0.43/0.80 | Adv2 loss/acc 1.33/0.51 | Cos Sim 0.19
> Epoch: 16 | Clf loss/acc 0.44/0.76 | Adv1 loss/acc 0.43/0.80 | Adv2 loss/acc 1.30/0.51 | Cos Sim 0.19
> Epoch: 17 | Clf loss/acc 0.44/0.76 | Adv1 loss/acc 0.43/0.80 | Adv2 loss/acc 1.29/0.51 | Cos Sim 0.18
> Epoch: 18 | Clf loss/acc 0.44/0.76 | Adv1 loss/acc 0.43/0.80 | Adv2 loss/acc 1.28/0.51 | Cos Sim 0.18
> Epoch: 19 | Clf loss/acc 0.43/0.77 | Adv1 loss/acc 0.43/0.80 | Adv2 loss/acc 1.27/0.51 | Cos Sim 0.17
> Epoch: 20 | Clf loss/acc 0.43/0.77 | Adv1 loss/acc 0.43/0.80 | Adv2 loss/acc 1.26/0.51 | Cos Sim 0.17
> Epoch: 21 | Clf loss/acc 0.43/0.77 | Adv1 loss/acc 0.43/0.80 | Adv2 loss/acc 1.25/0.51 | Cos Sim 0.16
> Epoch: 22 | Clf loss/acc 0.43/0.77 | Adv1 loss/acc 0.43/0.80 | Adv2 loss/acc 1.25/0.51 | Cos Sim 0.16
> Epoch: 23 | Clf loss/acc 0.43/0.77 | Adv1 loss/acc 0.43/0.80 | Adv2 loss/acc 1.24/0.51 | Cos Sim 0.15
> Epoch: 24 | Clf loss/acc 0.43/0.77 | Adv1 loss/acc 0.43/0.80 |

### EqOpp

In [ ]:
fairdef = "EqOpp"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvBin4EqOpp', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp]
    result += [a2_dp, a2_deqodds, a2_deqopp]
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

> Epoch: 1 | Clf loss/acc 0.77/0.28 | Adv1 loss/acc 0.18/0.81 | Adv2 loss/acc 0.55/0.51 | Cos Sim 0.09
> Epoch: 2 | Clf loss/acc 0.67/0.41 | Adv1 loss/acc 0.18/0.81 | Adv2 loss/acc 0.54/0.51 | Cos Sim 0.05
> Epoch: 3 | Clf loss/acc 0.62/0.67 | Adv1 loss/acc 0.19/0.81 | Adv2 loss/acc 0.53/0.51 | Cos Sim 0.00
> Epoch: 4 | Clf loss/acc 0.59/0.70 | Adv1 loss/acc 0.19/0.81 | Adv2 loss/acc 0.52/0.51 | Cos Sim 0.01
> Epoch: 5 | Clf loss/acc 0.58/0.72 | Adv1 loss/acc 0.19/0.81 | Adv2 loss/acc 0.50/0.51 | Cos Sim 0.03
> Epoch: 6 | Clf loss/acc 0.57/0.73 | Adv1 loss/acc 0.19/0.81 | Adv2 loss/acc 0.48/0.51 | Cos Sim 0.05
> Epoch: 7 | Clf loss/acc 0.56/0.74 | Adv1 loss/acc 0.19/0.81 | Adv2 loss/acc 0.46/0.51 | Cos Sim 0.06
> Epoch: 8 | Clf loss/acc 0.56/0.75 | Adv1 loss/acc 0.19/0.81 | Adv2 loss/acc 0.44/0.51 | Cos Sim 0.07
> Epoch: 9 | Clf loss/acc 0.56/0.75 | Adv1 loss/acc 0.19/0.81 | Adv2 loss/acc 0.42/0.51 | Cos Sim 0.08
> Epoch: 10 | Clf loss/acc 0.56/0.76 | Adv1 loss/acc 0.20/0.81 | Adv2 los

/Users/lffpl/Projects/falsb4mpa/falsb4mpa/evaluation/baseline_metrics.py:41: RuntimeWarning: invalid value encountered in scalar divide
  return pos(Y) / (pos(Y) + neg(Y))


> Epoch: 1 | Clf loss/acc 0.81/0.28 | Adv1 loss/acc 0.13/0.80 | Adv2 loss/acc 0.45/0.51 | Cos Sim 0.29
> Epoch: 2 | Clf loss/acc 0.69/0.41 | Adv1 loss/acc 0.13/0.80 | Adv2 loss/acc 0.44/0.51 | Cos Sim 0.24
> Epoch: 3 | Clf loss/acc 0.62/0.68 | Adv1 loss/acc 0.13/0.80 | Adv2 loss/acc 0.43/0.51 | Cos Sim 0.23
> Epoch: 4 | Clf loss/acc 0.57/0.70 | Adv1 loss/acc 0.13/0.80 | Adv2 loss/acc 0.42/0.51 | Cos Sim 0.29
> Epoch: 5 | Clf loss/acc 0.54/0.72 | Adv1 loss/acc 0.13/0.80 | Adv2 loss/acc 0.40/0.51 | Cos Sim 0.36
> Epoch: 6 | Clf loss/acc 0.52/0.73 | Adv1 loss/acc 0.13/0.80 | Adv2 loss/acc 0.38/0.51 | Cos Sim 0.41
> Epoch: 7 | Clf loss/acc 0.50/0.74 | Adv1 loss/acc 0.13/0.80 | Adv2 loss/acc 0.36/0.51 | Cos Sim 0.43
> Epoch: 8 | Clf loss/acc 0.49/0.75 | Adv1 loss/acc 0.12/0.80 | Adv2 loss/acc 0.34/0.51 | Cos Sim 0.45
> Epoch: 9 | Clf loss/acc 0.48/0.75 | Adv1 loss/acc 0.12/0.80 | Adv2 loss/acc 0.32/0.51 | Cos Sim 0.45
> Epoch: 10 | Clf loss/acc 0.47/0.76 | Adv1 loss/acc 0.12/0.80 | Adv2 los

/Users/lffpl/Projects/falsb4mpa/falsb4mpa/evaluation/baseline_metrics.py:41: RuntimeWarning: invalid value encountered in scalar divide
  return pos(Y) / (pos(Y) + neg(Y))


> Epoch: 1 | Clf loss/acc 0.81/0.28 | Adv1 loss/acc 0.15/0.80 | Adv2 loss/acc 0.39/0.51 | Cos Sim 0.03
> Epoch: 2 | Clf loss/acc 0.67/0.42 | Adv1 loss/acc 0.15/0.80 | Adv2 loss/acc 0.39/0.51 | Cos Sim -0.05
> Epoch: 3 | Clf loss/acc 0.59/0.67 | Adv1 loss/acc 0.15/0.80 | Adv2 loss/acc 0.38/0.51 | Cos Sim -0.09
> Epoch: 4 | Clf loss/acc 0.55/0.70 | Adv1 loss/acc 0.15/0.80 | Adv2 loss/acc 0.37/0.51 | Cos Sim -0.06
> Epoch: 5 | Clf loss/acc 0.52/0.72 | Adv1 loss/acc 0.15/0.80 | Adv2 loss/acc 0.36/0.51 | Cos Sim -0.02
> Epoch: 6 | Clf loss/acc 0.50/0.73 | Adv1 loss/acc 0.15/0.80 | Adv2 loss/acc 0.35/0.51 | Cos Sim 0.01
> Epoch: 7 | Clf loss/acc 0.49/0.74 | Adv1 loss/acc 0.15/0.80 | Adv2 loss/acc 0.34/0.51 | Cos Sim 0.03
> Epoch: 8 | Clf loss/acc 0.48/0.75 | Adv1 loss/acc 0.15/0.80 | Adv2 loss/acc 0.32/0.51 | Cos Sim 0.04
> Epoch: 9 | Clf loss/acc 0.47/0.75 | Adv1 loss/acc 0.15/0.80 | Adv2 loss/acc 0.32/0.51 | Cos Sim 0.05
> Epoch: 10 | Clf loss/acc 0.47/0.75 | Adv1 loss/acc 0.15/0.80 | Adv2

/Users/lffpl/Projects/falsb4mpa/falsb4mpa/evaluation/baseline_metrics.py:41: RuntimeWarning: invalid value encountered in scalar divide
  return pos(Y) / (pos(Y) + neg(Y))


> Epoch: 1 | Clf loss/acc 0.81/0.27 | Adv1 loss/acc 0.16/0.81 | Adv2 loss/acc 0.50/0.51 | Cos Sim 0.12
> Epoch: 2 | Clf loss/acc 0.70/0.41 | Adv1 loss/acc 0.16/0.81 | Adv2 loss/acc 0.49/0.51 | Cos Sim 0.05
> Epoch: 3 | Clf loss/acc 0.63/0.67 | Adv1 loss/acc 0.16/0.81 | Adv2 loss/acc 0.48/0.51 | Cos Sim 0.00
> Epoch: 4 | Clf loss/acc 0.58/0.71 | Adv1 loss/acc 0.17/0.81 | Adv2 loss/acc 0.47/0.51 | Cos Sim 0.02
> Epoch: 5 | Clf loss/acc 0.55/0.72 | Adv1 loss/acc 0.17/0.81 | Adv2 loss/acc 0.45/0.51 | Cos Sim 0.06
> Epoch: 6 | Clf loss/acc 0.53/0.73 | Adv1 loss/acc 0.17/0.81 | Adv2 loss/acc 0.42/0.51 | Cos Sim 0.08
> Epoch: 7 | Clf loss/acc 0.52/0.74 | Adv1 loss/acc 0.17/0.81 | Adv2 loss/acc 0.40/0.51 | Cos Sim 0.10
> Epoch: 8 | Clf loss/acc 0.50/0.75 | Adv1 loss/acc 0.17/0.81 | Adv2 loss/acc 0.38/0.51 | Cos Sim 0.11
> Epoch: 9 | Clf loss/acc 0.49/0.75 | Adv1 loss/acc 0.18/0.81 | Adv2 loss/acc 0.36/0.51 | Cos Sim 0.11
> Epoch: 10 | Clf loss/acc 0.49/0.76 | Adv1 loss/acc 0.18/0.81 | Adv2 los

## Saving into DF then CSV

In [ ]:
result_df = pd.DataFrame(results, columns=header)
result_df

,model_name,cv_seed,clas_acc,f1-micro,f1-macro,a1_dp,a1_deqodds,a1_deqopp,a2_dp,a2_deqodds,a2_deqopp,wc_spd,wc_aod,wc_eod,last_cosine_similarity
0,MultAdvBin4DP,13,0.761364,0.761364,0.683805,0.979545,0.985142,0.983830,0.692274,0.783637,0.692274,0.250000,0.166667,0.0,-0.169608
1,MultAdvBin4DP,29,0.758523,0.758523,0.673448,0.993486,0.980958,0.976992,0.967940,0.938515,0.967940,0.000000,0.000000,0.0,-0.113078
2,MultAdvBin4DP,42,0.772727,0.772727,0.691443,0.948778,0.952538,0.934006,1.000000,1.000000,1.000000,0.000000,0.000000,0.0,-0.250511
3,MultAdvBin4DP,55,0.750947,0.750947,0.664948,0.992062,0.951411,0.945978,0.590196,0.779622,0.590196,0.250000,0.500000,0.0,-0.221917
4,MultAdvBin4DP,73,0.771780,0.771780,0.685115,0.940117,0.965475,0.956223,1.000000,1.000000,1.000000,0.000000,0.000000,0.0,-0.084922
5,MultAdvBin4EqOdds,13,0.768939,0.768939,0.691529,0.986722,0.971135,0.960300,0.739708,0.807354,0.739708,0.250000,0.166667,0.0,-0.147321
6,MultAdvBin4EqOdds,29,0.776515,0.776515,0.692908,0.979248,0.993653,0.995114,0.988948,0.949020,0.988948,0.000000,0.000000,0.0,-0.028789
7,MultAdvBin4EqOdds,42,0.776989,0.776989,0.693350,0.962956,0.955292,0.924086,1.000000,1.000000,1.000000,0.000000,0.000000,0.0,-0.075046
8,MultAdvBin4EqOdds,55,0.760890,0.760890,0.675158,0.985874,0.968091,0.967288,0.590196,0.765336,0.590196,0.250000,0.500000,0.0,-0.193989
9,MultAdvBin4EqOdds,73,0.782670,0.782670,0.692204,0.927804,0.952171,0.940390,1.000000,1.000000,1.000000,0.000000,0.000000,0.0,-0.110359


In [22]:
result_df.to_csv(f'../../results/{data_name}-mult_adv-{epochs}.csv')